# Where-fi - Object localization using RSSI of Wi-fi signals indoors

### Project for the INTAR lecture at UPV

## Imports and configurations

### Imports of useful libraries

In [ ]:
!pip install -q torchmetrics
# Libraries for logging and general utilities
import random
import logging
import wandb
import time
import math
import yaml
from box.box import Box
import functools
import operator
import sys

# Libraries for data manipulation and analysis
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from ax.service.ax_client import AxClient, ObjectiveProperties

# Helpful libraries
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, Optional, Tuple, Union

# Deep learning libraries
import torch
from torch import nn
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
from torchmetrics.classification import (MulticlassAccuracy, MulticlassConfusionMatrix,
                                         MulticlassF1Score, MulticlassAUROC)
from torchmetrics.regression import (MeanAbsoluteError,
                                     MeanSquaredError)

### Configuration to control model architecture and training

In [ ]:
%%writefile Ensemble_MLP.yaml
name: "Ensemble_MLP"
classifier:
    input_dim: 520
    dropout: 0.4426632479755182
    floor_classes: 5
regressor:
    input_dim: 520
    dropout: 0.12498379261035723

loss_weights:
    classification: 0.31744355871521124
    regression: 1.0

In [ ]:
%%writefile Autoencoder.yaml
#TODO: [x] Set up the yaml file for the autoencoder model. 
name: "Autoencoder"
latent_dim: 64
classifier:
    dropout: 0.3
    floor_classes: 5
regressor:
    dropout: 0.3
loss_weights:
    classification: 1.0
    regression: 1.0
    decoder: 1.0

In [ ]:
%%writefile config.yaml
# Change 'local' to 'colab' if running in Google Colab, don't forget to adapt source path
environment: 'local'
# Seed for deterministics
seed: 42

# Number of classes for floor classification
floor_classes: 5
training:
    num_epochs: 61
    # Learning rate parameters
    learning_rate: 0.004721799506877059
    learning_rate_scheduler: "cosine"
    learning_rate_min: 0.0000041297
    # Optimizer parameters
    beta1: 0.9
    beta2: 0.999
    weight_decay: 0.01
logging:
    log_interval: 5
    eval_interval: 5
    # For WandB logging
    use_wandb: True
    team_name: "UPV_projects"
    project_name: "Where-fi"
    api_key: null
    # Add-on for the run name
    run_name_additional: None
use_model: "Ensemble_MLP"
#TODO:[x] Add Autoencoder use_model
#use_model: "Autoencoder"
dataset:
    batch_size: 64
    # For loading
    root_path: '/Users/timhager/Library/Mobile Documents/com~apple~CloudDocs/Masterstudium/Auslandssemester/INAR/project/dataset'
    training_data_name: "TrainingData.csv"
    validation_data_name: "ValidationData.csv"
    # For normalization & preprocessing
    no_signal_value: -110
    signal_max: 0
    longitude_min: -7695.9387549299299000
    longitude_max: -7299.786516730871000
    latitude_min: 4864745.7450159714
    latitude_max: 4865017.3646842018
    # For dataaugmentation
    data_augmentation:
        noise:
            enabled: True,
            noise_level: 0.091
            probability: 0.9
        dropout:
            enabled: True,
            probability: 0.05
        shift:
            enabled: True,
            max_shift: 0.0455
            probability: 0.7

In [ ]:
%%writefile optimization_config.yaml
active: False

optimization_config:
  objectives:
    classification_accuracy:
      minimize: False
      threshold: 0.90  # Optional: Min tolerated accuracy
    regression_mse:
      minimize: True
      threshold: 0.001  # Optional: Max tolerated MSE
  total_trials: 2

search_space:
  - name: "training.learning_rate"
    type: "range"
    bounds: [0.0001, 0.1]
    value_type: "float"
    log_scale: true  # Optional: Logarithmic scaling for LRs
  
  - name: "training.learning_rate_min"
    type: "range"
    bounds: [0.000001, 0.001]
    value_type: "float"
    log_scale: true  # Optional: Logarithmic scaling for LRs
    
  - name: "training.num_epochs"
    type: "range"
    bounds: [10, 100]
    value_type: "int"
  
  - name: "model.classifier.dropout"
    type: "range"
    bounds: [0.0, 0.5]
    value_type: "float"

  - name: "model.regressor.dropout"
    type: "range"
    bounds: [0.0, 0.5]
    value_type: "float"

  - name: "model.loss_weights.classification"
    type: "range"
    bounds: [0.0, 1.0]
    value_type: "float"

  - name: "model.loss_weights.regression"
    type: "range"
    bounds: [0.0, 1.0]
    value_type: "float"

In [ ]:
# To access by CONFIG.training.batch_size instead of CONFIG['training']['batch_size']
with open("config.yaml", "r") as f:
    CONFIG = Box(yaml.safe_load(f))
print(f"General configuration loaded from config.yaml")
with open(f"{CONFIG.use_model}.yaml", "r") as f:
    MODEL_CONFIG = Box(yaml.safe_load(f))
print(f"Model-specific configuration loaded for model {MODEL_CONFIG.name}")
CONFIG.model = MODEL_CONFIG
print(f"Configuration merged for model {CONFIG.model.name}")
with open(f"optimization_config.yaml", "r") as f:
    OPTIMIZATION_CONFIG = Box(yaml.safe_load(f))
    CONFIG.optimization = OPTIMIZATION_CONFIG
print(f"Optimization configuration loaded")


#### Support of Google Colab

In [ ]:
if CONFIG.environment == 'colab':
    from google.colab import drive
    drive.mount('/content/gdrive')
    dataset_path = 'gdrive/MyDrive/Colab Notebooks/where_fi/dataset'
    CONFIG.dataset.root_path = dataset_path
elif CONFIG.environment == 'local':
    pass
else:
  raise ValueError(f"Invalid environment '{CONFIG.environment}'. Must be one of 'local' or 'colab'.")

## Helper classes
Classes for utility functionalities

### Class for training parameters
Small helper class to organise the parameters for the training

In [ ]:
class TrainingParameters:
    '''
    Class to encapsulate training parameters.
    Contains learning rate, beta values, weight decay, and number of epochs.
    '''

    def __init__(self,
            lr: float,
            lr_min: float,
            lr_scheduler: str,
            beta1: float,
            beta2: float,
            weight_decay: float,
            epochs: int
        ):
        '''
        Args:
            lr (float): Learning rate
            beta1 (float): Beta1 value for optimizer
            beta2 (float): Beta2 value for optimizer
            weight_decay (float): Weight decay for optimizer
            epochs (int): Number of training epochs

        Initializes the training parameters with the given values.
        '''
        self.lr = lr
        self.lr_min = lr_min
        self.lr_scheduler = lr_scheduler
        self.beta1 = beta1
        self.beta2 = beta2
        self.weight_decay = weight_decay
        self.epochs = epochs

    def get_learning_rate(self) -> float:
        '''
        Get the learning rate.
        '''
        return self.lr

    def get_learning_rate_min(self) -> float:
        '''
        Get the minimum learning rate.
        '''
        return self.lr_min

    def get_betas(self) -> tuple[float, float]:
        '''
        Get the beta values.
        '''
        return (self.beta1, self.beta2)

    def get_weight_decay(self) -> float:
        '''
        Get the weight decay.
        '''
        return self.weight_decay

    def get_epochs(self) -> int:
        '''
        Get the number of epochs.
        '''
        return self.epochs

    def get_lr_scheduler(self) -> str:
        '''
        Get lr_schedular
        '''
        return self.lr_scheduler

## Dataset

### Dataaugmentation

In [ ]:
class GaussianNoise:
    '''
    Class to add Gaussian noise to the input signal.
    '''
    def __init__(self, std_range=(0.0, 0.1), p=0.0):
        """
        std: relative noise strength
        p: probability of applying the noise
        """
        self.std_range = std_range
        self.p = p

    def __call__(self, x):
        """
        x: (B, C)
        """
        if random.random() < self.p:
            std = random.uniform(self.std_range[0], self.std_range[1])
            noise = torch.randn_like(x) * std
            # Maximal noise of given level
            noise = torch.clamp(noise, -0.1, 0.1)
            result = x + noise
            # Make sure the result is still between 0 and 1
            result = torch.clamp(result, 0.0, 1.0)
            return result
        return x

In [ ]:
class RandomValueDropout:
    '''
    Class to randomly drop (set to zero) some values in the input signal.
    '''
    def __init__(self, p=0.0):
        '''
        p: probability of dropping a value.
        '''
        self.p = p

    def __call__(self, x):
        '''
        Args:
            x (torch.Tensor): Input tensor of shape (B, C).
        Returns:
            torch.Tensor: Tensor with some values dropped.
        '''
        if random.random() < self.p:
            mask = torch.rand_like(x) > 0.9
            x = x.masked_fill(mask, 0.0)
        return x

In [ ]:
class Shifting:
    '''
    Class to add a shift to the input signal.
    '''
    def __init__(self, shift=0.05, p=0.0):
        """
        shift: relative shift strength
        p: probability of applying the shift
        """
        self.shift = shift
        self.p = p

    def __call__(self, x):
        """
        x: (B, C)
        """
        if random.random() < self.p:
            shift = random.uniform(-self.shift, self.shift)
            # Apply the shift
            result = x + shift
            # Make sure the result is still between 0 and 1
            result = torch.clamp(result, 0.0, 1.0)
            return result
        return x

In [ ]:
class DataAugmentation:
    """
    Class for randomly applying data augmentations of the following list:
        - Noise
        - Null masking of RSS values
        - Bias shifting of RSS values
    """
    def __init__(self, cfg: any):
        '''
        Initializes the DataAugmentation with a composition of random
        transformations.
        '''
        self.transform = transforms.Compose([
            GaussianNoise(std_range=(0.0, cfg.data_augmentation.noise.noise_level), p=cfg.data_augmentation.noise.probability),
            Shifting(shift=cfg.data_augmentation.shift.max_shift, p=cfg.data_augmentation.shift.probability),
            RandomValueDropout(p=cfg.data_augmentation.dropout.probability)
        ])

    def __call__(self, x: torch.Tensor) -> torch.Tensor:
        '''
        Args:
            x (torch.Tensor): Input tensor to be augmented.
        Returns:
            torch.Tensor: Augmented tensor.

        Method to apply the random data augmentations to the input signal.
        '''
        return self.transform(x)

### Dataset

In [ ]:
class ProjectDataset(Dataset):
    """
    Dataset class for loading ...
    """
    def __init__(self, cfg: Any,
                 split: str,
                 sample_transform: transforms = None,
                 label_transform: transforms = None
        ) -> None:
        """
        Initialize the dataset.

        Args:
            cfg: Configuration object with dataset paths and parameters.
            split: One of 'train', 'val', or 'test'.
            scaler: Optional dictionary {'mean': ..., 'std': ...} for normalization.
        """
        self.split = split
        self.random_seed = cfg.seed
        self.cfg = cfg.dataset
        self.data_root = Path(self.cfg.root_path)
        self.sample_transform = sample_transform
        self.label_transform = label_transform


        self.training_data_path = self.data_root / self.cfg.training_data_name
        self.test_data_path = self.data_root / self.cfg.validation_data_name

        # Check if files exist
        if not self.training_data_path.exists():
            raise FileNotFoundError(f"Training data file not found at {self.training_data_path}")
        if not self.test_data_path.exists():
            raise FileNotFoundError(f"Test data file not found at {self.test_data_path}")


        self.samples, self.labels = self._load_data()

        if len(self.samples) == 0:
            raise RuntimeError(
                f"Dataset split '{split}' is empty. "
            )


    def _load_data(self) -> Tuple[np.ndarray, np.ndarray]:
        """
        Load raw sensor files, merge them, and segment into datapoints.

        Returns:
            Tuple of (samples, labels) where samples is (N, Window, 6).
        """
        # Remove unnecessary columns
        drop_columns = ["RELATIVEPOSITION", "USERID", "PHONEID", "TIMESTAMP"]

        if self.split == 'test':
            # Read test data
            test_data = pd.read_csv(self.test_data_path)
            data = test_data.drop(columns=drop_columns)

        elif self.split in ['training', 'validation']:
            # Read training & validation data
            training_data = pd.read_csv(self.training_data_path)
            data = training_data.drop(columns=drop_columns)
        else:
            raise ValueError(f"Invalid split '{self.split}'. Must be one of 'training', 'validation', or 'test'.")


        # Replace no signal values with a specific value (e.g., -110 dBm)
        data = data.replace(100, self.cfg.no_signal_value)

        # Normalize
        columns_to_normalize = data.columns.difference(['FLOOR', 'BUILDINGID', 'LATITUDE', 'LONGITUDE', 'SPACEID'])
        max_value = 0
        min_value = self.cfg.no_signal_value
        data[columns_to_normalize] = (data[columns_to_normalize] - min_value) / (max_value - min_value)

        # Organize labels
        # Append building ID and floor ID to create a unique label for each location
        data['CLASSID'] = data['BUILDINGID'].astype(str) + '_' + data['SPACEID'].astype(str)
        data = data.drop(columns=['BUILDINGID', 'SPACEID'])

        # Check whether longitude or lattitude values are outside of the expected range and raise an error if so
        if (data['LONGITUDE'] < self.cfg.longitude_min).any() or (data['LONGITUDE'] > self.cfg.longitude_max).any():
            raise ValueError("Longitude values are outside of the expected range.")
        if (data['LATITUDE'] < self.cfg.latitude_min).any() or (data['LATITUDE'] > self.cfg.latitude_max).any():
            raise ValueError("Latitude values are outside of the expected range.")

        # Normalize longitude and latitude to be between 0 and 1
        data['LATITUDE'] = (data['LATITUDE'] - self.cfg.latitude_min) / (self.cfg.latitude_max - self.cfg.latitude_min)
        data['LONGITUDE'] = (data['LONGITUDE'] - self.cfg.longitude_min) / (self.cfg.longitude_max - self.cfg.longitude_min)


        if self.split in ['training', 'validation']:
            # Split into training & validation data
            training_data, validation_data = train_test_split(
                data,
                test_size=0.10,           # 10% for validation
                random_state=self.random_seed,          # For deterministic
                stratify=data['CLASSID']  # Guarantees same representation of classes
            )

            if self.split == 'training':
                data = training_data
            else:
                data = validation_data


        label_columns = ["FLOOR", "LONGITUDE", "LATITUDE", "CLASSID"]
        labels = data.loc[:, label_columns].copy()
        samples = data.drop(columns=label_columns)

        return np.array(samples, dtype=np.float32), np.array(labels, dtype=np.float32)


    def __len__(self) -> int:
        """
        Return the number of valid windows in the dataset.
        """
        return len(self.samples)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, Union[int, torch.Tensor]]:
        """
        Return a window and its label.

        Output shapes:
        - seq2label: (Window_Size, 6), Scalar Integer
        - seq2seq:   (Window_Size, 6), (Window_Size,) (LongTensor)
        """
        x = torch.from_numpy(self.samples[idx])
        y = torch.from_numpy(self.labels[idx])

        if self.sample_transform:
            x = self.sample_transform(x)
        if self.label_transform:
            y = self.label_transform(y)

        return x, y

## Class for general utilities

In [ ]:
class Utilities:
    '''
    Class to encapsulate utilities for training, such as logging.
    '''

    def __init__(self):
        '''
        Class to organize different utility functionalities.
        '''
        pass

    def seed_worker(self, worker_id):
        '''
        Seed worker for DataLoader to ensure reproducibility.
        '''
        _ = worker_id  # Unused, but required by the DataLoader
        worker_seed = torch.initial_seed() % 2**32
        np.random.seed(worker_seed)
        random.seed(worker_seed)

    def set_by_path(self, nested_box, path, value):
        keys = path.split('.')
        functools.reduce(operator.getitem, keys[:-1], nested_box)[keys[-1]] = value

    def get_dataset(self, cfg, split='training'):
        '''
        Args:
            cfg: Config object
            split: 'training', 'validation', or 'test'

        Return:
            dataset
            Dataloader
        '''
        if split == 'training':
            transform_ops = [
                DataAugmentation(cfg.dataset)
            ]
        else:  # no augmentation for test
            transform_ops = [
            ]

        datapoint_transform = transforms.Compose(transform_ops)
        label_transform = transforms.Compose([])


        dataset = ProjectDataset(cfg, split, sample_transform=datapoint_transform,
                        label_transform=label_transform)


        # Generator for reproducibility
        g = torch.Generator()
        g.manual_seed(cfg.seed)

        dataloader = DataLoader(
            dataset,
            batch_size=cfg.dataset.batch_size,
            shuffle=(split == 'training'), # Just shuffle during training
            worker_init_fn=self.seed_worker,
            generator=g
        )

        return dataset, dataloader


    def set_seed(self, seed: int):
        '''
        Set random seed for reproducibility across all libraries.

        Args:
            seed (int): Random seed value
        '''
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)  # For multi-GPU
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

    def get_parameters_table(self, model: torch.nn.Module, position: int = 0) -> str:
        '''
        Args:
            model (torch.nn.Module): The PyTorch model to summarize
            position (int): The position in the model hierarchy (recursive depth)

        Returns a list of tuples containing (module name, position, number of trainable parameters).
        '''
        # name, position, number trainable parameters
        lines = []
        num_trainable_params = 0
        for _, param in model.named_parameters(recurse=False):
            if param.requires_grad:
                num_trainable_params += math.prod(param.size())

        child_lines = []
        for module in model.children():
            child_lines += self.get_parameters_table(module, position + 1)

        sum_params = 0
        for line in child_lines:
            if line[1] == position + 1:
                sum_params += line[2]

        if num_trainable_params == 0:
            lines.append([type(model).__name__, position, sum_params])
        else:
            lines.append([type(model).__name__, position, num_trainable_params])
        lines += child_lines

        return lines

    def get_model_summary(self, model: torch.nn.Module) -> str:
        '''
        Args:
            model (torch.nn.Module): The PyTorch model to summarize

        Returns a string representation of the model's summary.
        '''
        lines = self.get_parameters_table(model)
        summary = ""
        for line in lines:
            summary += "  " * line[1] + f"└─ {line[0]}: {line[2]} trainable parameters\n"
        summary += "=================================================================\n"
        summary += f"Total Trainable Parameters: {lines[0][2]}\n"
        return summary
    
    def calc_meter_from_mae(self, cfg: Any, mae_lat: float, mae_lon: float) -> float:
        '''
        Convert Mean Absolute Error (MAE) to a custom meter value for better interpretability.

        Args:
            cfg (Any): Configuration object containing the necessary parameters for conversion.
            mae_lat (float): The Mean Absolute Error for latitude.
            mae_lon (float): The Mean Absolute Error for longitude.

        Returns:
            float: The corresponding meter value.
        '''
        # Set min & max values
        lat_min, lat_max = cfg.dataset.latitude_min, cfg.dataset.latitude_max
        lon_min, lon_max = cfg.dataset.longitude_min, cfg.dataset.longitude_max

        # Denormalisation
        mae_lat_meters = mae_lat * (lat_max - lat_min)
        mae_lon_meters = mae_lon * (lon_max - lon_min)

        total_mae_meters = np.sqrt(mae_lat_meters**2 + mae_lon_meters**2)

        return total_mae_meters

## Evaluator class
Class for handling the evaluation of the model with the test set, calculating different relevant metrics

In [ ]:
class Evaluator:
    """
    Evaluator for binary and multiclass classification tasks.

    Handles model evaluation with metrics computation for both:
        - Binary classification: num_classes=1 (BCELoss) or num_classes=2 (CrossEntropyLoss)
        - Multiclass classification: num_classes>2 (CrossEntropyLoss)

    Automatically selects metrics (Accuracy, Precision, Recall, F1, AUROC, Confusion Matrix)
    based on the classification task type.
    """

    def __init__(
        self,
        cfg: Any,
        eval_loader: DataLoader,
        model: nn.Module,
        device: torch.device,
        wb_run: Any

    ):
        """
        Args:
            cfg: Hydra config. Must contain 'num_classes'.
            eval_loader (DataLoader): DataLoader for the evaluation dataset.
            model (nn.Module): The model to evaluate.
            device (torch.device): 'cuda' or 'cpu'.
            criterion (nn.Module, optional): Loss function (nn.BCELoss or nn.CrossEntropyLoss).
                                            Uses 'nn.CrossEntropyLoss' as default.
        """
        self.cfg = cfg
        self.eval_loader = eval_loader
        self.model = model
        self.device = device
        self.wb_run = wb_run

        # Set loss function
        if self.model.get_name() == "Ensemble_MLP":
            self.criterion_classification = torch.nn.CrossEntropyLoss()
            self.criterion_regression = torch.nn.MSELoss()
            self.classification_loss_weight = cfg.model.loss_weights.classification
            self.regression_loss_weight = cfg.model.loss_weights.regression
        elif self.model.get_name() == "Autoencoder":
            #TODO:[x] Set up the loss function for the autoencoder, do we need a additional loss function for the Encoder - Decoder case?
            self.criterion_classification = torch.nn.CrossEntropyLoss()
            self.criterion_regression = torch.nn.MSELoss()
            self.criterion_decoder = torch.nn.MSELoss()
            self.classification_loss_weight = cfg.model.loss_weights.classification
            self.regression_loss_weight = cfg.model.loss_weights.regression
            self.decoder_loss_weight = cfg.model.loss_weights.decoder

        # Raise error if cfg doesnt have 'num_classes' attribute
        if not hasattr(cfg, 'floor_classes'):
            raise ValueError(
                "Config object 'cfg' must contain 'floor_classes' attribute.")
        self.num_classes = cfg.floor_classes

        # Define a dict containing "task" and "num_classes" for intializing the metrics
        self.task_kwargs = {"num_classes": self.num_classes}

        # Creates and fills a dict to store all metrics
        self.metrics = nn.ModuleDict()
        self.metrics.update(self._get_metrics())
        for metric in self.metrics.values():
            metric.to(device)

    def _get_metrics(self) -> dict:
        """
        Build the dictionary of evaluation metrics.

        Regression metrics:
            - MSE
            - MAE

        Multiclass metrics:
            - Accuracy (micro-averaged)
            - F1, AUROC (macro-averaged)
            - Confusion Matrix

        Returns:
            dict: Dictionary mapping metric names to torchmetrics instances.
        """
        # Metrics for multiclass classification and regression
        macro_kwargs = {**self.task_kwargs, "average": "macro"}
        micro_kwargs = {**self.task_kwargs, "average": "micro"}
        metrics = {
            'conf_matrix': MulticlassConfusionMatrix(**self.task_kwargs),
            'accuracy_evaluation': MulticlassAccuracy(**micro_kwargs),
            'auroc_macro': MulticlassAUROC(**macro_kwargs),
            'f1_macro': MulticlassF1Score(**macro_kwargs),
            'mse': MeanSquaredError(),
            'mae': MeanAbsoluteError(),
            'mae_lat': MeanAbsoluteError(),
            'mae_lon': MeanAbsoluteError()
        }

        return metrics

    def _reset_metrics(self) -> None:
        """
        Resets all metric states. Called at the beginning of eval.
        """
        for metric in self.metrics.values():
            metric.reset()

    @torch.no_grad()
    def eval(self, return_metrics: bool = False) -> None:
        """
        Args:
            return_metrics: Optionally return metric. Default: False

        Run a full evaluation loop over the evaluation dataset and logs the metrics to
        Weights & Biases.
        """
        self.model.eval()
        self._reset_metrics()

        running_loss = 0.0

        for _, (x, y_true) in enumerate(self.eval_loader):
            x, y_true = x.to(self.device), y_true.to(self.device)
            y_floor = y_true[:, 0].long()
            y_coordinates = y_true[:, 1:3]

            # Make a prediction
            if self.model.get_name() == "Ensemble_MLP":
                y_pred_classification, y_pred_regression = self.model(x)

                y_pred_classification_probs = torch.softmax(y_pred_classification, dim=1)
                y_pred_classification_labels = torch.argmax(y_pred_classification, dim=1)

                loss_classification = self.criterion_classification(y_pred_classification, y_floor)
                loss_regression = self.criterion_regression(y_pred_regression, y_coordinates)

                loss = self.classification_loss_weight * loss_classification + self.regression_loss_weight * loss_regression
            elif self.model.get_name() == "Autoencoder":
                #TODO: [x] Will we have two types of predictions one with Encoder - Decoder and one with the regression and classifier head?
                y_pred_classification, y_pred_regression, y_pred_decoder = self.model(x)

                y_pred_classification_probs = torch.softmax(y_pred_classification, dim=1)
                y_pred_classification_labels = torch.argmax(y_pred_classification, dim=1)

                loss_classification = self.criterion_classification(y_pred_classification, y_floor)
                loss_regression = self.criterion_regression(y_pred_regression, y_coordinates)
                loss_decoder = self.criterion_decoder(y_pred_decoder, x)

                loss = self.classification_loss_weight * loss_classification + self.regression_loss_weight * loss_regression + self.decoder_loss_weight * loss_decoder
            else:
                raise NotImplementedError(f"Model {self.model.get_name()} not implemented in training loop.")

            running_loss += loss.item()

            # Iterate through the dict to update the metrics
            for metric_name, metric in self.metrics.items():
                # Multiclass metrics
                if isinstance(metric, MulticlassAUROC):
                    # auroc uses probability instead of labels
                    metric.update(y_pred_classification_probs, y_floor)
                elif isinstance(metric, MulticlassConfusionMatrix) or isinstance(metric, MulticlassAccuracy) or isinstance(metric, MulticlassF1Score):
                    metric.update(y_pred_classification_labels, y_floor)
                # Regression metrics
                elif isinstance(metric, MeanSquaredError) or isinstance(metric, MeanAbsoluteError):
                    if metric_name == 'mae_lon':
                        metric.update(y_pred_regression[:, 0].contiguous(), y_coordinates[:, 0].contiguous())
                    elif metric_name == 'mae_lat':
                        metric.update(y_pred_regression[:, 1].contiguous(), y_coordinates[:, 1].contiguous())
                    else:
                        metric.update(y_pred_regression, y_coordinates.contiguous())
                else:
                    logging.error(f"Metric {metric_name} not implemented in evaluation loop.")

        # Compute final metrics for hole dataset after iterating through all batches
        final_metrics = {}

        # Calculate average loss per batch
        final_metrics = {
            'total_loss_evaluation': running_loss / len(self.eval_loader)}

        # Compute all other metrics
        for metric_name, metric in self.metrics.items():
            try:
                final_metrics[metric_name] = metric.compute()
            except Exception as e:
                logging.warning("[Warning] Could not compute metric '%s'. Set to None. Error: %s",
                                metric_name, e)
                final_metrics[metric_name] = None
        print(f"Evaluation accuracy: {final_metrics['accuracy_evaluation'].item()}")

        # Calculate the custom meter value from the MAE of latitude and longitude
        mae_lat = final_metrics['mae_lat'].item()
        mae_lon = final_metrics['mae_lon'].item()
        util = Utilities()
        final_metrics['mae_meters'] = util.calc_meter_from_mae(self.cfg, mae_lat, mae_lon)
        print(f"Evaluation MAE in meters: {final_metrics['mae_meters']}")

        # Remove mae_lat and mae_lon from final_metrics
        del final_metrics['mae_lat']
        del final_metrics['mae_lon']

        if self.wb_run is not None:
            self.wb_run.log(final_metrics)

        if return_metrics:
            return final_metrics

## Trainer class
Class for handling the training of the model

In [ ]:
class Trainer:
    '''
    Class to handle the training loop for a model.
    '''

    def __init__(
        self,
        cfg: any, # Contains the configuration parameters for training, such as learning rate, epochs, etc.
        train_loader: DataLoader,
        model: nn.Module,
        evaluator: Evaluator,
        device: torch.device,
        wb_run: Any
    ):
        '''
        Args:
            cfg (object): Configuration object with training parameters.
            train_loader (DataLoader): DataLoader for the training dataset.
            model (nn.Module): The model to be trained.
            evaluator (Evaluator): Evaluator for validation during training.
            device (torch.device): Device to run the training on.

        Initializes the Trainer with the given arguments.
        '''
        self.train_loader = train_loader

        # Ogranize training parameters, so not all are attributes of this class
        self.training_parameters = TrainingParameters(
            lr=cfg.training.learning_rate,
            lr_min=cfg.training.learning_rate_min,
            lr_scheduler=cfg.training.learning_rate_scheduler,
            beta1=cfg.training.beta1,
            beta2=cfg.training.beta2,
            weight_decay=cfg.training.weight_decay,
            epochs=cfg.training.num_epochs,
        )

        # Create optimizer
        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=self.training_parameters.get_learning_rate(),
            betas=self.training_parameters.get_betas(),
            weight_decay=self.training_parameters.get_weight_decay()
        )

        # Initialize training components
        model.to(device)
        self.model = model
        self.optimizer = optimizer
        self.evaluator = evaluator
        self.device = device

        # Set intervals
        self.log_interval = cfg.logging.log_interval
        self.eval_interval = cfg.logging.eval_interval

        # Set loss function
        if self.model.get_name() == "Ensemble_MLP":
            self.criterion_classification = torch.nn.CrossEntropyLoss()
            self.criterion_regression = torch.nn.MSELoss()
            self.classification_loss_weight = cfg.model.loss_weights.classification
            self.regression_loss_weight = cfg.model.loss_weights.regression
        elif self.model.get_name() == "Autoencoder":
        #TODO:[x] Set up the loss function for the autoencoder, do we need a additional loss function for the Encoder - Decoder case?
            self.criterion_classification = torch.nn.CrossEntropyLoss()
            self.criterion_regression = torch.nn.MSELoss()
            self.criterion_decoder = torch.nn.MSELoss()
            self.classification_loss_weight = cfg.model.loss_weights.classification
            self.regression_loss_weight = cfg.model.loss_weights.regression
            self.decoder_loss_weight = cfg.model.loss_weights.decoder

        # Change learning rate scheduler based on config
        self.scheduler = None
        if self.training_parameters.lr_scheduler == "cosine":
            self.scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
                self.optimizer,
                T_max=self.training_parameters.get_epochs(),
                eta_min=self.training_parameters.get_learning_rate_min()
            )

        self.wb_run = wb_run
        self.cfg = cfg


    def get_trained_model(self) -> nn.Module:
        '''
        Returns:
            The trained model

        Function to access the trained model by the Trainer class.
        '''
        return self.model

    def train(self):
        '''
        Main training loop.
        '''

        #TODO: Adjust for Autoencoder, training (freeze after some epoch...)
        for epoch in range(self.training_parameters.get_epochs()):
            start_time = time.time()
            # Set model to training mode
            self.model.train()
            running_loss = 0
            correct = 0
            total = 0

            for step, (x, y_true) in enumerate(self.train_loader):
                x, y_true = x.to(self.device), y_true.to(self.device)
                y_floor = y_true[:, 0].long()
                y_coordinates = y_true[:, 1:3]
                # TODO: Remove after debugging
                #print(y_coordinates.size())

                self.optimizer.zero_grad()

                if self.model.get_name() == "Ensemble_MLP":
                    y_pred_classification, y_pred_regression = self.model(x)
                    loss_classification = self.criterion_classification(y_pred_classification, y_floor)
                    loss_regression = self.criterion_regression(y_pred_regression, y_coordinates)

                    loss = self.classification_loss_weight * loss_classification + self.regression_loss_weight * loss_regression
                elif self.model.get_name() == "Autoencoder":
                    #TODO: [x] Will we have two types of predictions one with Encoder - Decoder and one with the regression and classifier head?
                    y_pred_classification, y_pred_regression, y_pred_decoder = self.model(x)

                    loss_classification = self.criterion_classification(y_pred_classification, y_floor)
                    loss_regression = self.criterion_regression(y_pred_regression, y_coordinates)
                    loss_decoder = self.criterion_decoder(y_pred_decoder, x)

                    loss = self.classification_loss_weight * loss_classification + self.regression_loss_weight * loss_regression + self.decoder_loss_weight * loss_decoder
                else:
                    raise NotImplementedError(f"Model {self.model.get_name()} not implemented in training loop.")

                loss.backward()

                # Gradient clipping
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)

                self.optimizer.step()

                running_loss += loss.item() * x.size(0)

                predicted_classes = torch.argmax(y_pred_classification, dim=1)
                total += y_floor.size(0)
                correct += (predicted_classes == y_floor).sum().item()


                if self.wb_run is not None and ((step + 1) % self.log_interval == 0):
                    accuracy = correct / total if total > 0 else 0
                    # Log the base learning rate (first param group, likely head)
                    current_lr = self.optimizer.param_groups[0]['lr']

                    self.wb_run.log({
                        "floor_accuracy_training": accuracy,
                        "coordinates_mse_training": loss_regression.item(),
                        "total_loss_training": loss.item(),
                        "classification_loss_training": loss_classification.item(),
                        "regression_loss_training": loss_regression.item(),
                        "learning_rate": current_lr
                    })

            end_time = time.time()
            if self.wb_run is not None:
                self.wb_run.log({"epoch_duration": end_time - start_time})

            if self.scheduler is not None:
                self.scheduler.step()

            print("Epoch [{0}/{1}]".format(epoch+1, self.training_parameters.get_epochs()))

            if (epoch + 1) % self.eval_interval == 0:
                # Set model to evaluation mode
                self.model.eval()
                with torch.no_grad():
                    self.evaluator.eval()

        logging.info("Training finished")
        # For optimization the metrics have to be returned
        return_metrics = self.cfg.optimization.active
        if return_metrics:
            return self.evaluator.eval(return_metrics)

## Model
Class of the selected model including the definition of the architecture as well as the forward porapagation

### Simple MLP
Simple MLP model for comparison baseline and to test code early during development.

In [ ]:
class Classifier(nn.Module):
    '''
    MLP to classify the Floor..
    '''
    def __init__(self, cfg: Any):
        '''
        Initializes the model based on the configuration.
        '''
        super().__init__()
        self.name = "Classifier"
        self.layers = nn.Sequential(
            nn.Linear(cfg.input_dim, 256),
            nn.BatchNorm1d(256),
            nn.LeakyReLU(),
            nn.Dropout(cfg.dropout),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.LeakyReLU(),
            nn.Linear(128, cfg.floor_classes)
        )

    def get_name(self) -> str:
        '''
        Returns the name of the model.
        '''
        return self.name

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        '''
        Forward pass of the model.

        Args:
            x: Input tensor

        Returns:
            Output tensor after passing through the model.
        '''
        x = self.layers(x)
        # Softmax for classification output
        x = torch.softmax(x, dim=1)
        return x

class Regressor(nn.Module):
    '''
    MLP to regress the longitude and latitude.
    '''
    def __init__(self, cfg: Any):
        '''
        Initializes the model based on the configuration.
        '''
        super().__init__()
        self.name = "Regressor"
        self.layers = nn.Sequential(
            nn.Linear(cfg.input_dim, 256),
            nn.BatchNorm1d(256),
            nn.LeakyReLU(),
            nn.Dropout(cfg.dropout),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.LeakyReLU(),
            nn.Linear(128, 2)
        )

    def get_name(self) -> str:
        '''
        Returns the name of the model.
        '''
        return self.name

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        '''
        Forward pass of the model.

        Args:
            x: Input tensor

        Returns:
            Output tensor after passing through the model.
        '''
        x = self.layers(x)
        # Ensure output is between 0 and 1 for normalized coordinates
        x = torch.sigmoid(x)
        return x


class Ensemble_MLP(nn.Module):
    '''
    Exemple for a simple MLP model consisting of a classification for the FLoor and a regression for the longitude and latitude.
    '''
    def __init__(self, cfg: Any):
        '''
        Initializes the model based on the configuration.
        '''
        super().__init__()
        self.name = "Ensemble_MLP"

        self.classifier = Classifier(cfg.classifier)
        self.regressor = Regressor(cfg.regressor)

    def get_name(self) -> str:
        '''
        Returns the name of the model.
        '''
        return self.name

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        '''
        Forward pass of the hybrid model.

        Args:
            x: Input tensor

        Returns:
            Output tensor after passing through the model.
        '''

        y_pred_classification = self.classifier(x)
        y_pred_regression = self.regressor(x)

        return y_pred_classification, y_pred_regression

### Autoencoder
Autoencoder model based on paper "Indoor Localization With an Autoencoder-Based
Convolutional Neural Network" by Hatice Arslantas and Selcuk Okdem.

In [ ]:
#TODO: Implement Autoencoder class architecture, Endcoder, Decoder, Regressor, Classifier

## Main function

In [ ]:
def main():
    '''
    Main function to run the training and evaluation pipeline.
    '''
    cfg = CONFIG
    if cfg is None:
        raise ValueError("Config dictionary must be provided.")

    # Create utility instance
    util = Utilities()

    # Set seed for reproducibility
    util.set_seed(cfg.seed)

    # Select device (GPU if available, else CPU)
    device = torch.device('cpu')
    if torch.cuda.is_available():
        device = torch.device('cuda')
    elif torch.backends.mps.is_available():
        device = torch.device('mps')
    logging.info('Using device: %s', device)

    # Get Datasets
    _, train_dataloader = util.get_dataset(cfg, split='training')
    _, eval_dataloader = util.get_dataset(cfg, split='validation')
    _, test_dataloader = util.get_dataset(cfg, split='test')

    # Get Model
    if cfg.model.name == "Ensemble_MLP":
        model = Ensemble_MLP(cfg.model)
    elif cfg.model.name == "Autoencoder":
        #TODO: Add Autoencoder Architecture class
        pass
    else:
        raise NotImplementedError(f"Model {cfg.model.name} not implemented in training loop.")

    model_summary = util.get_model_summary(model)
    print(model_summary)

    # Start a new wandb run to track this script
    if cfg.logging.use_wandb:
        timestamp = datetime.now().strftime('%Y%m%d_%H%M')
        wandb.login(key=cfg.logging.api_key)
        additional = cfg.logging.run_name_additional
        if additional != 'None' and additional is not None:
            run_name = f"{model.get_name()}_{additional}_{timestamp}"
        else:
            run_name = f"{model.get_name()}_{timestamp}"
        wb_run = wandb.init(
            # Set the wandb entity where your project will be logged
            entity=cfg.logging.team_name,
            # Set the wandb project where this run will be logged
            project=cfg.logging.project_name,
            # Name of the run
            name=run_name,
            # Track hyperparameters and run metadata
            config={
                "learning_rate": cfg.training.learning_rate,
                "learning_rate scheduler": cfg.training.learning_rate_scheduler,
                "architecture": model.get_name(),
                "dataset": "UjiIndoorLoc",
                "epochs": cfg.training.num_epochs,
                # Log all model config settings
                "model settings": {**dict(cfg.model)},
                "model architecture": model_summary
            }
        )
    else:
        wb_run = None


    # Get evaluator
    evaluator = Evaluator(cfg, eval_dataloader, model, device, wb_run)

    # Get trainer
    trainer = Trainer(cfg,
                      train_dataloader,
                      model,
                      evaluator,
                      device,
                      wb_run)

    # Train the model and perform final evaluation on test set
    trainer.train()

    test_evaluator = Evaluator(cfg, test_dataloader, trainer.get_trained_model(), device, wb_run)
    print("Running Evaluation")
    test_evaluator.eval()

    print("Finished")
#TODO: Calculate the meter out of the 0...1 prediction coordinates



## Bayesian optimizer
To optimize the hyperparameters of the model and achieve better performance

In [ ]:
def optimize():
    '''
    Main function to run the training and evaluation pipeline.
    '''
    cfg = CONFIG
    if cfg is None:
        raise ValueError("Config dictionary must be provided.")

    # Create utility instance
    util = Utilities()

    # Set seed for reproducibility
    util.set_seed(cfg.seed)

    # Select device (GPU if available, else CPU)
    device = torch.device('cpu')
    if torch.cuda.is_available():
        device = torch.device('cuda')
    elif torch.backends.mps.is_available():
        device = torch.device('mps')
    logging.info('Using device: %s', device)

    # Get Datasets
    _, train_dataloader = util.get_dataset(cfg, split='training')
    _, eval_dataloader = util.get_dataset(cfg, split='validation')
    _, test_dataloader = util.get_dataset(cfg, split='test')


    # Initialize AxClient for hyperparameter optimization
    ax_client = AxClient()

    ax_objectives = {
        name: ObjectiveProperties(
            minimize=props["minimize"], 
            threshold=props.get("threshold")
        )
    for name, props in cfg.optimization.optimization_config.objectives.items()
}

    # Create experiment with data from optimization configuration
    ax_client.create_experiment(
        name="bayesian_optimization",
        parameters=cfg.optimization.search_space,
        objectives=ax_objectives,
        overwrite_existing_experiment=True
    )

    # 4. Der Bayesian Optimization Loop
    for i in range(cfg.optimization.optimization_config.total_trials):
        print(f"Starting Trial {i + 1}/{cfg.optimization.optimization_config.total_trials}...")
        
        # Ax schlägt die nächsten optimalen Parameter vor (Bayesian State)
        parameters, trial_index = ax_client.get_next_trial()
        
        # Adapt parameters in the config for the current trial
        for param_name, param_value in parameters.items():
            util.set_by_path(cfg, param_name, param_value)

        # Get Model with adapted parameters
        if cfg.model.name == "Ensemble_MLP":
            model = Ensemble_MLP(cfg.model)
        else:
            raise NotImplementedError(f"Model {cfg.model.name} not implemented in training loop.")
        
        model_summary = util.get_model_summary(model)
        logging.info(model_summary)

        # Start a new wandb run to track this script
        if cfg.logging.use_wandb:
            timestamp = datetime.now().strftime('%Y%m%d_%H%M')
            wandb.login(key=cfg.logging.api_key)
            additional = cfg.logging.run_name_additional
            if additional != 'None' and additional is not None:
                run_name = f"{model.get_name()}_{additional}_{timestamp}"
            else:
                run_name = f"{model.get_name()}_{timestamp}"
            wb_run = wandb.init(
                # Set the wandb entity where your project will be logged
                entity=cfg.logging.team_name,
                # Set the wandb project where this run will be logged
                project=cfg.logging.project_name,
                # Name of the run
                name=run_name,
                # Track hyperparameters and run metadata
                config={
                    "learning_rate": cfg.training.learning_rate,
                    "learning_rate scheduler": cfg.training.learning_rate_scheduler,
                    "architecture": model.get_name(),
                    "dataset": "UjiIndoorLoc",
                    "epochs": cfg.training.num_epochs,
                    # Log all model config settings
                    "model settings": {**dict(cfg.model)},
                    "model architecture": model_summary
                }
            )
        else:
            wb_run = None

        # Get evaluator
        evaluator = Evaluator(cfg, eval_dataloader, model, device, wb_run)

        # Get trainer
        trainer = Trainer(cfg,
                        train_dataloader,
                        model,
                        evaluator,
                        device,
                        wb_run)
        
        # Train the model and perform evaluation on validation set
        metrics = trainer.train()

        raw_data = {"classification_accuracy": metrics['accuracy_evaluation'].item(),
                    "regression_mse": metrics['mse'].item()}
        
        # Ergebnisse an Ax zurückmelden (wichtig für das Update des Gauß-Prozesses)
        ax_client.complete_trial(trial_index=trial_index, raw_data=raw_data)

    # 5. Beste Parameter ausgeben
    pareto_results = ax_client.get_pareto_optimal_parameters()
    print("\n--- Optimization complete ---")
    print(f"Best parameters: {pareto_results[0][0]}")
    print(f"Expected performance: {pareto_results[0][1]}")
    print("\n\n")
    print("Training & evaluating best model on test set")

    # Final evaluation on test set with the best found parameters

    # Adapt parameters in the config for the current trial
    for param_name, param_value in pareto_results[0][0].items():
        util.set_by_path(cfg, param_name, param_value)

    # Get Model with adapted parameters
    if cfg.model.name == "Ensemble_MLP":
        model = Ensemble_MLP(cfg.model)
    else:
        raise NotImplementedError(f"Model {cfg.model.name} not implemented in training loop.")
    
    # Get evaluator
    evaluator = Evaluator(cfg, eval_dataloader, model, device, wb_run)

    # Get trainer
    trainer = Trainer(cfg,
                    train_dataloader,
                    model,
                    evaluator,
                    device,
                    wb_run)
    
    trainer.train()

    test_evaluator = Evaluator(cfg, test_dataloader, trainer.get_trained_model(), device, wb_run)
    logging.info("Running Evaluation")
    test_evaluator.eval()



In [ ]:
if __name__ == '__main__':
    if CONFIG.optimization.active:
        optimize()
    else:
        main()